# AmsiNet - Step 4: MDIW-13 index, document-level splits, and zero-shot transfer from BSTD

**Runtime:** Colab, any GPU (T4 is enough; the A100 is not needed). Needs `MyDrive/amsinet/data/MultiscriptDatabase.zip`, `bstd_index.json`, and the BSTD word agents `agents/word_seed0.pt` and `agents/word_moe_seed0.pt` from Step 3a.

What this notebook does
1. Unzips MDIW-13 and builds a **document -> line -> word** index from the file names (the hierarchy is annotated in the names, e.g. `gurm_052.png`, `gurm_052_007.png`, `gurm_052_007_004.png`).
2. Keeps the ten scripts that overlap with our BSTD label set (Hindi = Devanagari, Bangla = Bengali-Assamese, Gujrati = Gujarati, Oriya = Odia, Roman = Latin) and drops Arabic, Japanese and Thai.
3. Splits **by document** (60/20/20 per script), so words of one document never appear in two splits, and saves the index to Drive.
4. Evaluates the BSTD-trained word agents on MDIW-13 words **without any training on MDIW-13** (zero-shot transfer from scene text to scanned documents).

Every document in MDIW-13 contains one script. Page, line and word labels therefore always agree, and this dataset tests transfer and early exit, not mixed-script routing.

In [ ]:
# 1. Setup
from google.colab import drive
import os
if not os.path.ismount('/content/drive'): drive.mount('/content/drive')
!pip -q install timm
import torch, timm
print('torch', torch.__version__, '| timm', timm.__version__, '| GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE (switch runtime to GPU for step 6)')

In [ ]:
# 2. Configuration and unzip
import os, json, glob, math, random, time, subprocess, collections
import numpy as np, pandas as pd
from PIL import Image
ROOT, LOCAL = '/content/drive/MyDrive/amsinet', '/content/mdiw'
ZIP = f'{ROOT}/data/MultiscriptDatabase.zip'
CFG = dict(DOMAIN='Printed',        # 'Printed' (main) or 'Handwritten' (secondary; Gujarati has only 3 documents)
           VERSION='Original',      # 'Original' = raw scans, '' = processed copy; keep one version for all experiments
           SPLIT=(0.6, 0.2, 0.2), SEED=0, BATCH=256, WORKERS=8)
# folder name -> our BSTD script name (Arabic, Japanese and Thai are not in our label set)
FOLDER2SCRIPT = {'Bangla': 'Bengali-Assamese', 'Hindi': 'Devanagari', 'Gujrati': 'Gujarati', 'Gurmukhi': 'Gurmukhi', 'Kannada': 'Kannada',
                 'Malayalam': 'Malayalam', 'Oriya': 'Odia', 'Roman': 'Latin', 'Tamil': 'Tamil', 'Telugu': 'Telugu'}
os.makedirs(f'{ROOT}/mdiw', exist_ok=True)
if not glob.glob(f'{LOCAL}/**/*.png', recursive=True):
    subprocess.run(['unzip', '-q', '-o', ZIP, '-d', LOCAL], check=True)
BASE = os.path.dirname(glob.glob(f'{LOCAL}/**/Multiscript{CFG["DOMAIN"]}Words{CFG["VERSION"]}', recursive=True)[0])
print('base folder:', BASE, '| domain', CFG['DOMAIN'], '| version', CFG['VERSION'] or 'processed')

In [ ]:
# 3. Build the document -> line -> word index from the file names
def scan(level):
    rows = []
    d = f'{BASE}/Multiscript{CFG["DOMAIN"]}{level}{CFG["VERSION"]}'
    for folder in sorted(os.listdir(d)):
        if folder not in FOLDER2SCRIPT: continue
        for f in sorted(os.listdir(f'{d}/{folder}')):
            if not f.lower().endswith('.png'): continue
            parts = os.path.splitext(f)[0].lower().split('_')
            rows.append(dict(path=f'{d}/{folder}/{f}', script=FOLDER2SCRIPT[folder], parts=parts))
    return pd.DataFrame(rows)

DOCS, LINES, WORDS = scan('Documents'), scan('Lines'), scan('Words')
for df, n in ((DOCS, 2), (LINES, 3), (WORDS, 4)):
    bad = df[df.parts.map(len) != n]
    assert len(bad) == 0, f'unexpected file names: {bad.path.head().tolist()}'
    df['doc'] = df.parts.map(lambda p: '_'.join(p[:2]))
    if n >= 3: df['line'] = df.parts.map(lambda p: '_'.join(p[:3]))
    df.drop(columns='parts', inplace=True)

# consistency of the hierarchy
orph_l = (~LINES.doc.isin(set(DOCS.doc))).sum(); orph_w = (~WORDS.line.isin(set(LINES.line))).sum(); orph_wd = (~WORDS.doc.isin(set(DOCS.doc))).sum()
print(f'documents {len(DOCS)} | lines {len(LINES)} | words {len(WORDS)}')
print(f'lines without a document: {orph_l} | words without a line: {orph_w} | words without a document: {orph_wd}')
print('prefix check (script -> file-name prefix):', {s: sorted(set(DOCS[DOCS.script == s].doc.str.split('_').str[0])) for s in FOLDER2SCRIPT.values()})

In [ ]:
# 4. Split by document (stratified per script), save the index
rng = np.random.RandomState(CFG['SEED']); DOCSPLIT = {}
for s in sorted(set(DOCS.script)):
    ds = sorted(set(DOCS[DOCS.script == s].doc) | set(LINES[LINES.script == s].doc) | set(WORDS[WORDS.script == s].doc))
    ds = list(rng.permutation(ds)); n = len(ds)
    nte = max(1, round(CFG['SPLIT'][2] * n)); nva = max(1, round(CFG['SPLIT'][1] * n))
    for i, d in enumerate(ds): DOCSPLIT[d] = 'test' if i < nte else ('val' if i < nte + nva else 'train')
for df in (DOCS, LINES, WORDS): df['split'] = df.doc.map(DOCSPLIT)
tag = CFG['DOMAIN'].lower()
DOCS.to_csv(f'{ROOT}/mdiw/{tag}_docs.csv', index=False); LINES.to_csv(f'{ROOT}/mdiw/{tag}_lines.csv', index=False); WORDS.to_csv(f'{ROOT}/mdiw/{tag}_words.csv', index=False)
t = pd.concat({'documents': DOCS.groupby(['script', 'split']).size().unstack(1), 'lines': LINES.groupby(['script', 'split']).size().unstack(1),
               'words': WORDS.groupby(['script', 'split']).size().unstack(1)}, axis=1).fillna(0).astype(int)
display(t)
print('words per script (all splits):'); print(WORDS.script.value_counts().to_string())
print('check: no document in two splits ->', all(DOCS.groupby('doc').split.nunique() == 1))

In [ ]:
# 5. Look at the data: one document, one line and four words for three scripts
import matplotlib.pyplot as plt
fig, axs = plt.subplots(3, 6, figsize=(20, 7))
for r, s in enumerate(['Devanagari', 'Tamil', 'Latin']):
    d = DOCS[DOCS.script == s].iloc[3]; l = LINES[LINES.doc == d.doc].iloc[1]; w = WORDS[WORDS.line == l.line].head(4)
    items = [d.path, l.path] + w.path.tolist()
    for ax, p in zip(axs[r], items + [None] * (6 - len(items))):
        ax.axis('off')
        if p: ax.imshow(Image.open(p)); ax.set_title(os.path.basename(p), fontsize=8)
plt.tight_layout(); plt.show()

In [ ]:
# 6. Zero-shot transfer: BSTD-trained word agents evaluated on all MDIW-13 words (no MDIW training)
import torch.nn as nn, torchvision.transforms as T
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import f1_score
D = json.load(open(f'{ROOT}/data/bstd_index.json', encoding='utf-8')); SCRIPTS = D['scripts']; SIDX = {s: i for i, s in enumerate(SCRIPTS)}; C = len(SCRIPTS)
M_EXPERTS, FILL, DEV = 4, (124, 116, 104), 'cuda' if torch.cuda.is_available() else 'cpu'

def letterbox(im, w, h):
    iw, ih = im.size; s = min(w / iw, h / ih); nw, nh = max(1, int(round(iw * s))), max(1, int(round(ih * s)))
    im = im.resize((nw, nh), Image.BICUBIC, reducing_gap=3.0); cv = Image.new('RGB', (w, h), FILL); cv.paste(im, ((w - nw) // 2, (h - nh) // 2)); return cv

class MoEHead(nn.Module):
    def __init__(s, d, ncls, M, hid=256):
        super().__init__(); s.gate = nn.Linear(d, M)
        s.experts = nn.ModuleList([nn.Sequential(nn.Linear(d, hid), nn.ReLU(), nn.Linear(hid, ncls)) for _ in range(M)])
    def forward(s, f):
        g = s.gate(f).softmax(-1); z = torch.stack([e(f) for e in s.experts], 1); return (g.unsqueeze(-1) * z).sum(1), g

class WordAgent(nn.Module):
    def __init__(s, M=None, name='efficientvit_b0.r224_in1k'):
        super().__init__(); s.bb = timm.create_model(name, pretrained=False, num_classes=0); s.bb.eval()
        with torch.no_grad(): d = s.bb(torch.zeros(2, 3, 224, 224)).shape[1]
        s.head = nn.Linear(d, C) if M is None else MoEHead(d, C, M)
    def forward(s, x):
        f = s.bb(x); o = s.head(f); return o if isinstance(o, tuple) else (o, None)

mc = timm.data.resolve_data_config({}, model=timm.create_model('efficientvit_b0.r224_in1k', pretrained=False, num_classes=0))
tf = T.Compose([T.ToTensor(), T.Normalize(mc['mean'], mc['std'])])
class WDS(Dataset):
    def __init__(s, paths): s.p = list(paths)
    def __len__(s): return len(s.p)
    def __getitem__(s, i): return tf(letterbox(Image.open(s.p[i]).convert('RGB'), 224, 224))

@torch.no_grad()
def predict(model, paths):
    model.eval(); out = []
    for x in DataLoader(WDS(paths), batch_size=CFG['BATCH'], num_workers=CFG['WORKERS']):
        with torch.autocast(DEV, dtype=torch.float16): lo, _ = model(x.to(DEV))
        out.append(lo.float().softmax(1).cpu().numpy())
    return np.concatenate(out)

y = WORDS.script.map(SIDX).values; ZS = {}
for tagm, M in (('word_seed0', None), ('word_moe_seed0', M_EXPERTS)):
    model = WordAgent(M).to(DEV); model.load_state_dict(torch.load(f'{ROOT}/agents/{tagm}.pt', map_location=DEV)); t0 = time.time()
    P = predict(model, WORDS.path); ZS[tagm] = P; pred = P.argmax(1)
    np.save(f'{ROOT}/mdiw/zeroshot_{tagm}_{tag}_probs.npy', P)
    te = (WORDS.split == 'test').values
    print(f'{tagm}: ALL words acc {100 * (pred == y).mean():.2f} macro-F1 {100 * f1_score(y, pred, average="macro"):.2f} | '
          f'test-split words acc {100 * (pred[te] == y[te]).mean():.2f} macro-F1 {100 * f1_score(y[te], pred[te], average="macro"):.2f} ({time.time() - t0:.0f}s)')
WORDS[['path', 'script', 'doc', 'line', 'split']].to_csv(f'{ROOT}/mdiw/zeroshot_{tag}_paths.csv', index=False)

In [ ]:
# 7. Per-script F1 and the most frequent confusions (plain word agent)
from sklearn.metrics import confusion_matrix, precision_recall_fscore_support
P = ZS['word_seed0']; pred = P.argmax(1)
pr, rc, f1, sup = precision_recall_fscore_support(y, pred, labels=range(C), zero_division=0)
tab = pd.DataFrame({'script': SCRIPTS, 'F1': (100 * f1).round(1), 'recall': (100 * rc).round(1), 'precision': (100 * pr).round(1), 'words': sup})
display(tab.sort_values('F1'))
Cm = confusion_matrix(y, pred, labels=range(C)); off = [(Cm[i, j] / Cm[i].sum(), SCRIPTS[i], SCRIPTS[j], Cm[i, j]) for i in range(C) for j in range(C) if i != j and Cm[i].sum() > 0]
print('top confusions (true -> predicted, share of true words):')
for r, a, b, n in sorted(off, reverse=True)[:12]: print(f'  {a:>17} -> {b:<17} {100 * r:5.1f}%  ({n})')

## Next steps
Paste back (1) the file counts and the split table, (2) the two `acc / macro-F1` lines, (3) the per-script table. Then **disconnect and delete the runtime**.